In [14]:
import pandas as pd

from scipy.optimize import brentq

Código CRI - 24E2453531
Devedora - FGR Incorporações
Emissão - 
Classe - Sênior
Data de Emissão - 29/05/2024
Data de Vencimento - 18/06/2041
Valor Nominal Inicial - R$ 185.000.000,00
Valor Nominal Unitário Inicial - R$ 1.000,00
Indexador - IPCA + 9,00% a.a.
Taxa de juros da emissão - 9,00% a.a.
Amortização - Mensal
Data-base do cálculo - 22/06/2026
Taxa de desconto usada no exercício - taxa de emissão + 1,00% a.a.

In [5]:
df_fluxo = pd.read_excel('./../../data/dados_tratados/df_fluxo.xlsx')

In [8]:
def calcular_pu(df, taxa_desconto, data_base, coluna_data="data", coluna_fluxo="fluxo"):
    df_calc = df.copy()

    df_calc[coluna_data] = pd.to_datetime(df_calc[coluna_data])
    data_base = pd.to_datetime(data_base)

    # pega apenas fluxos futuros
    df_calc = df_calc[df_calc[coluna_data] > data_base].copy()

    df_calc["dias"] = (df_calc[coluna_data] - data_base).dt.days

    df_calc["fator_desconto"] = (1 + taxa_desconto) ** (df_calc["dias"] / 365)

    df_calc["vp_fluxo"] = df_calc[coluna_fluxo] / df_calc["fator_desconto"]

    pu = df_calc["vp_fluxo"].sum()

    return pu

In [11]:
def calcular_taxa_implicita(
    df,
    pu_alvo,
    data_base,
    coluna_data="data",
    coluna_fluxo="fluxo",
    taxa_min=-0.99,
    taxa_max=1.00
):
    def erro(taxa):
        pu_calculado = calcular_pu(
            df=df,
            taxa_desconto=taxa,
            data_base=data_base,
            coluna_data=coluna_data,
            coluna_fluxo=coluna_fluxo
        )
        return pu_calculado - pu_alvo

    taxa = brentq(erro, taxa_min, taxa_max)

    return taxa

In [17]:
data_base = pd.to_datetime("2026-06-22")
taxa_desconto = 0.10  # 10% a.a.

# pega apenas fluxos futuros
df_pu = df_fluxo[df_fluxo["data"] > data_base].copy()

# garante fluxo sem NaN
df_pu["fluxo"] = df_pu["fluxo"].fillna(0)

# dias corridos até cada pagamento
df_pu["dias_corridos"] = (
    df_pu["data"] - data_base
).dt.days

# fator de desconto em dias corridos
df_pu["fator_desconto"] = (
    1 + taxa_desconto
) ** (df_pu["dias_corridos"] / 365)

# valor presente do fluxo
df_pu["vp_fluxo"] = (
    df_pu["fluxo"] / df_pu["fator_desconto"]
)

# PU na data-base
pu = df_pu["vp_fluxo"].sum()

In [20]:
df_pu

,Unnamed: 0,data,juros,amortizacao,fluxo,dias_corridos,fator_desconto,vp_fluxo
25,25,2026-07-15,7.228057,2.803471,10.031528,23,1.006024,9.971461
26,26,2026-08-15,7.208129,2.752138,9.960267,54,1.014201,9.820806
27,27,2026-09-15,6.948126,2.771974,9.720100,85,1.022444,9.506734
28,28,2026-10-15,7.168863,2.864220,10.033083,115,1.030485,9.736276
29,29,2026-11-15,6.909403,2.812842,9.722245,146,1.038860,9.358570
...,...,...,...,...,...,...,...,...
200,200,2041-02-15,0.316299,9.747570,10.063869,5352,4.045214,2.487846
201,201,2041-03-15,0.282280,9.829664,10.111944,5380,4.074899,2.481520
202,202,2041-04-15,0.205304,9.887857,10.093161,5411,4.108018,2.456941
203,203,2041-05-15,0.142123,9.961651,10.103774,5441,4.140326,2.440333


In [23]:
pu

948.8831471437176